# 03 · Эксперименты NBCO

**Пайплайн НИР — стадия 3 из 3.** Улучшение существующей сети через
NBCO (learned construction + trim/extend edit-политика). Основные результаты статьи:

- **единичный прогон** (dry-run плана + один запуск);
- **alpha-sweep** (E1: neural BCO vs Our NBCO по компромиссу RTT↔WMC);
- **E2 Pareto** (alpha × adj_target);
- **ablation** с разным набором пчёл (E2 5-model);
- **MACSA Table B**.

Case study на реальной сети (Екатеринбург) — в `04_case_study.ipynb`.

Всё параметризуется в коде через `build_experiment(...)` / `run_experiment(...)`
(M015): город, alpha, adj_target, число итераций, границы маршрута — аргументами,
`None` ⇒ значение из YAML. Логика — в `search/` и `paper_experiments/`.

In [ ]:
import pandas as pd
from IPython.display import display

from connectpt.routes_generator.core import (
    load_experiment, build_experiment, ExperimentRunFactory)
from connectpt.routes_generator.paper_experiments.paper_runs import (
    run_experiment, run_batch, save_results)
from connectpt.routes_generator import render_report

## Куда пишем и сколько считаем

Два параметра ниже — то же, что флаги `--out-dir` и `-p n_iterations` у
`scripts/run_nbco.py`: папка результатов и бюджет BCO. `ITERS=None` берёт бюджет
из YAML эксперимента (полный бюджет статьи).

In [ ]:
# Куда писать результаты и с каким бюджетом (то же, что --out-dir / -p n_iterations
# у scripts/run_nbco.py). ITERS=None -> бюджет из YAML эксперимента.
OUT_DIR = 'artifacts/results/notebook_03'
ITERS = 2      # короткий прогон; для полного бюджета статьи поставьте None
print('результаты ->', OUT_DIR, '| итераций BCO:', ITERS or 'из конфига')

## Единичный прогон

Сначала — **dry-run плана** (состав пчёл + какие модели грузятся, без BCO-цикла),
затем один запуск Our NBCO на Mandl при фиксированном alpha.

In [ ]:
cfg = build_experiment('table3_nbco_vs_our', bee_sets='our_nbco', models='construction_and_edit_seeded', city='Mandl', alpha=0.5, n_iterations=ITERS)
plan = ExperimentRunFactory.from_cfg(cfg).run(dry_run=True)
print('модели   :', plan.metadata['models_loaded'])
print('пчёлы    :', plan.plan['counts'])

single = run_experiment('table3_nbco_vs_our', out_dir=OUT_DIR, bee_sets='our_nbco', models='construction_and_edit_seeded', city='Mandl', alpha=0.5, n_iterations=ITERS)
single.display()

## Alpha-sweep — Table 3 (neural BCO vs Improved NBCO)

`run_batch("table3_nbco_vs_our", out_dir=..., city=...)` гоняет оба метода по
сетке alpha (компромисс время-в-пути ↔ связность) и строит сравнительную таблицу
(`tab:nbco-vs-our-only-with-init`, τ=0.3, 200 iter) + Pareto. Один batch = один
артефакт статьи; внутри — по прогону на метод.

In [ ]:
e1 = run_batch('table3_nbco_vs_our', out_dir=OUT_DIR, city='Mandl', n_iterations=ITERS)
e1.display()

## Pareto-фронт — Table 4 / Figure 4 (adjustment-target sweep, Mumford0)

Двумерный свип Improved NBCO на Mumford0: компромисс качества и степени изменения сети
(`adj_target`).

In [ ]:
e2 = run_experiment('table4_fig4_our_pareto', out_dir=OUT_DIR, n_iterations=ITERS)
e2.display()

## Ablation — Table 5 / Figure 5 (пять операторных комбинаций, Mumford1)

Пять вариантов пчелиного состава (GNN/RPC × trim-extend/type2 + trim12+extend12)
на Mumford1, adjustment выключен. Набор пчёл и модели приходят из групп
`search/bee_sets/*` + `search/models/*` — один батч, разные листья.

In [ ]:
e2_5 = run_batch('table5_fig5_5model', out_dir=OUT_DIR, n_iterations=ITERS)
e2_5.display()

## MACSA · Table B

Скоринг фиксированных маршрутов Table-B + alpha-sweep Our NBCO + выбор лучшего
решения, побеждающего MACSA. Один вызов; `sweep_name` выбирает свип: iter=1
(самостоятельная таблица статьи) или полный бюджет.

In [ ]:
from connectpt.routes_generator.paper_experiments.macsa_run import (
    run_macsa_table_b, SWEEP_EXPERIMENT, SWEEP_EXPERIMENT_ITER1)

# SWEEP_EXPERIMENT_ITER1 — одношаговый свип (самостоятельная таблица статьи);
# SWEEP_EXPERIMENT — полный бюджет (100 итераций на точку).
macsa = run_macsa_table_b(out_dir=OUT_DIR, sweep_name=SWEEP_EXPERIMENT_ITER1)
macsa.display()

## Итог

Все таблицы, дампы маршрутов и фигуры сохранены в `OUT_DIR`. Полный бюджет
статьи — `ITERS = None` (гоняет пользователь). Case study Екатеринбурга —
`04_case_study.ipynb`.